In [1]:
import numpy as np
from numpy.linalg import norm
import math, random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
#spatial funcs for WoS

def closestPoint(v, s):
    u = s[1] - s[0]
    rate = max(0, min(u.dot(v - s[0])/u.dot(u), 1))
    return (s[0] + rate*(s[1] - s[0]))

def shortestDistance(v, segments):
    r = float("inf")
    for s in segments:
        u = closestPoint(v, s)
        if norm(v - u) < r:
            r = norm(v - u)
    return r

In [12]:
def solver(v, segments, g, eps = 0.01, nWalks = 100, maxSteps = 16):
    vWalks = 0   
    sumEst = 0
    for i in range(0, nWalks):
        x0 = v
        for step in range(0, maxSteps): 
            #r = min([shortestDistance(x0, segments), maxR])
            r = shortestDistance(x0, segments)
            if r < eps: 
                sumEst += g(x0)
                vWalks += 1
                #print("walk: " + str(vWalks) + " hit " + str(x0) + " boundary " + str(g(x0)))
                break
            theta = random.uniform(0, 2*math.pi)
            x0 = x0 + np.array([r * math.cos(theta), r * math.sin(theta)])
    if vWalks == 0:
        return 0
    return sumEst/vWalks

In [13]:
# set up the problem
segments = [np.array([[-1, -1], [-1, 1]]), np.array([[-1, -1], [1, -1]]), np.array([[1, -1], [1, 1]]), np.array([[-1, 1], [1, 1]])]
#v = np.array([0, 0])
def boundary(v):
    return v[1] * v[0]

In [17]:
target = []
width = range(1, 100)
height = range(1, 100)
for i in width:
    for j in height:
        target.append(np.array([i/50 - 1, j/50 - 1]))

print(len(target))

9801


In [18]:
result = []
for v in target:
    result.append(solver(v, segments, boundary))

print(result[100])

0.9350527578839399


In [93]:
print(result[2024])
print(target[2024])

0.005894421412317938
[-0.58 -0.1 ]


In [52]:
target1 = np.array(target).reshape(-1, 2).astype("float32")
result1 = np.array(result)
#result1.shape

In [98]:
YNNmodel = keras.Sequential( # a list of fully connected layers
    [
        keras.Input(shape = (2)), #not necessary, but for model.summary
        keras.layers.Dense(256, activation = 'tanh'),
        keras.layers.Dense(128, activation = 'tanh'),
        keras.layers.Dense(64, activation = 'tanh'),
        keras.layers.Dense(1), #no softmax activation
    ]
)
print(YNNmodel.summary())


Model: "sequential_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_28 (Dense)            (None, 256)               768       
                                                                 
 dense_29 (Dense)            (None, 128)               32896     
                                                                 
 dense_30 (Dense)            (None, 64)                8256      
                                                                 
 dense_31 (Dense)            (None, 1)                 65        
                                                                 
Total params: 41985 (164.00 KB)
Trainable params: 41985 (164.00 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [111]:
YNNmodel.compile( # netwrok configurations
    loss = keras.losses.MeanSquaredError(), # do the output softmax
    optimizer = keras.optimizers.Adam(lr = 1e-4),
    #metrics = ["accuracy"],
)

YNNmodel.fit(target1, result1, batch_size = 64, epochs = 5, verbose = 2) # concrete training of the network

Epoch 1/5
154/154 - 1s - loss: 0.0016 - 1s/epoch - 10ms/step
Epoch 2/5
154/154 - 1s - loss: 0.0015 - 834ms/epoch - 5ms/step
Epoch 3/5
154/154 - 1s - loss: 0.0014 - 845ms/epoch - 5ms/step
Epoch 4/5
154/154 - 1s - loss: 0.0015 - 843ms/epoch - 5ms/step
Epoch 5/5
154/154 - 1s - loss: 0.0015 - 860ms/epoch - 6ms/step


In [112]:
YNNmodel(np.array([-1, 1]).reshape(1, 2))

<tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[-0.9656424]], dtype=float32)>